In [1]:
import csv
import os
import json
import pandas as pd
import re
import shutil
import time

from collections import defaultdict, deque, Counter
from dotenv import load_dotenv
from typing import List
from pydantic import BaseModel, Field

from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

from google import genai


# Load environment variables
load_dotenv(override=True)

MODEL= os.getenv("MODEL")
MODEL_SUPERVISOR= os.getenv("MODEL_SUPERVISOR")
MODEL_EMBEDDING = os.getenv("MODEL_EMBEDDING")
API_KEY = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=API_KEY)

BASE_DIR = os.path.dirname(os.getcwd())
DOCUMENT_FOLDER = os.path.join(BASE_DIR, "data/full_text_pdf")
VECTOR_DB_PATH = os.path.join(BASE_DIR, "vectorstore", "pdf_index")

# Remove previous FAISS index if present
if os.path.exists(VECTOR_DB_PATH):
    print(f"[CLEANUP] Removed existent vectorstore: {VECTOR_DB_PATH}")
    shutil.rmtree(VECTOR_DB_PATH)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [2]:
#for root, dirs, files in os.walk(BASE_DIR):
#    for file in files:
#        if file.lower().endswith(".pdf"):
#            src_path = os.path.join(root, file)
#            dst_path = os.path.join(BASE_DIR, file)
#
#            if os.path.abspath(src_path) != os.path.abspath(dst_path):
#                if os.path.exists(dst_path):
#                    name, ext = os.path.splitext(file)
#                    i = 1
#                    while os.path.exists(os.path.join(BASE_DIR, f"{name}_{i}{ext}")):
#                        i += 1
#                    dst_path = os.path.join(BASE_DIR, f"{name}_{i}{ext}")
#
#                shutil.move(src_path, dst_path)
#                print(f"Transfer: {src_path} → {dst_path}")

#for root, dirs, files in os.walk(BASE_DIR, topdown=False):
#    for d in dirs:
#        dir_path = os.path.join(root, d)
#        shutil.rmtree(dir_path)
#        print(f"Deleted Directory: {dir_path}")

In [11]:
RUN_ID = "run_5"

In [12]:
# Embedding and text split configuration

CHUNK_SIZE = 2000
CHUNK_OVERLAP = 400
MAX_CONTEXT_CHARS = 30000

# Simple rate-limiting mechanism
_RATE_WINDOW_SEC = 60.0
_RATE_MAX_CALLS = 10
_rate_calls_log = deque()

_sent_calls_count = 0
_total_input_tokens = 0
_total_output_tokens = 0
_total_embedding_tokens = 0


def _rate_limit_block_until_allowed():
    """Blocks execution to respect Gemini API rate limits."""
    while True:
        now = time.monotonic()

        while _rate_calls_log and (now - _rate_calls_log[0]) >= _RATE_WINDOW_SEC:
            _rate_calls_log.popleft()

        if len(_rate_calls_log) < _RATE_MAX_CALLS:
            _rate_calls_log.append(now)
            return

        time.sleep(0.5)


# Pydantic response schemas
class ExtractedDatasets(BaseModel):
    datasets: List[str] = Field(description="List of official AQA dataset names found")

class HarmonizedSources(BaseModel):
    sources: List[str] = Field(description="Harmonized list of dataset names in input order")


def truncate_context(text: str, limit_chars: int) -> str:
    """Truncates long text while preserving character encoding."""
    if len(text) <= limit_chars:
        return text
    return text[:limit_chars - 3] + "..."

def clean_text(text: str) -> str:
    """Removes invalid UTF-8 characters."""
    return text.encode("utf-8", "ignore").decode("utf-8", "ignore")


# FAISS Index Construction
def build_or_load_index(document_folder: str, vector_db_path: str):
    """Builds or loads the FAISS index for PDF documents."""
    embedding_model = GoogleGenerativeAIEmbeddings(model=MODEL_EMBEDDING, google_api_key=API_KEY)

    if os.path.exists(vector_db_path):
        try:
            vs = FAISS.load_local(vector_db_path, embedding_model, allow_dangerous_deserialization=False)
            print(f"[FAISS] Loaded index from {vector_db_path}. Total vectors: {vs.index.ntotal}")
            return vs, vs.as_retriever(), vs.index.ntotal
        except Exception as e:
            print(f"[FAISS] Error loading index ({e}), rebuilding index.")

    # Load and chunk PDFs
    pdf_files = [f for f in os.listdir(document_folder) if f.endswith(".pdf")]
    all_docs = []

    for file in pdf_files:
        path = os.path.join(document_folder, file)
        loader = PyPDFLoader(path)
        pages = loader.load()

        print(f"[LOAD] {file}: {len(pages)} pages")

        for p in pages:
            p.metadata["source"] = file

        all_docs.extend(pages)

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    documents = text_splitter.split_documents(all_docs)

    # Clean encoding issues
    for d in documents:
        try:
            d.page_content.encode("utf-8")
        except UnicodeEncodeError as e:
            print(f"[WARN] Unicode error in {d.metadata.get('source')}: {e}")
        d.page_content = clean_text(d.page_content)

    global _total_embedding_tokens
    all_chunk_texts = [d.page_content for d in documents]
    count_resp = client.models.count_tokens(
        model=MODEL_EMBEDDING,
        contents=all_chunk_texts
    )
    _total_embedding_tokens = count_resp.total_tokens

    # Show chunk distribution per PDF
    counter = Counter(d.metadata['source'] for d in documents)
    for pdf_name, num_chunks in counter.items():
        print(f"[SPLIT] {pdf_name}: {num_chunks} chunks created")

    # Build FAISS index
    vectorstore = FAISS.from_documents(documents, embedding_model)
    total_chunks = vectorstore.index.ntotal

    print(f"[FAISS] Created vectors: {total_chunks}")

    os.makedirs(vector_db_path, exist_ok=True)
    vectorstore.save_local(vector_db_path)

    print(f"[FAISS] Saved index to {vector_db_path}")
    return vectorstore, vectorstore.as_retriever(), total_chunks


# Dataset extraction
def extract_datasets_per_pdf(retriever, query: str, total_chunks: int = 100, top_chunks_per_paper: int = 5):
    """Retrieves chunks via RAG and extracts AQA dataset names via LLM."""
    global _sent_calls_count, _total_input_tokens, _total_output_tokens

    k = total_chunks // 3
    fetch_k = total_chunks // 2

    results = retriever.vectorstore.max_marginal_relevance_search(
        query, k=k, fetch_k=fetch_k, lambda_mult=0.9
    )

    print(f"[RAG] Retrieved results: {len(results)} chunks")

    # Group chunks per PDF
    groups = defaultdict(list)
    for doc in results:
        src = doc.metadata.get("source", "Unknown")
        groups[src].append(doc.page_content)

    paper_datasets = {}

    # Process each PDF
    for source, chunks in groups.items():
        selected_chunks = chunks[:top_chunks_per_paper]
        combined = "\n\n---\n\n".join(selected_chunks)
        context = truncate_context(combined, MAX_CONTEXT_CHARS)

        prompt = (
            "You are an assistant that analyzes full scientific papers.\n"
            "Given the following extracted text from a paper, extract ONLY the official and recognized names of datasets that satisfy the following conditions:\n"
            "(a) The dataset is specifically used for Human Action Quality Assessment (AQA). This condition must be strictly respected.\n"
            "(b) Include only real, unique, and identifiable resources (dataset, database, or data collection).\n"
            "(c) Do NOT write generic descriptions (e.g., 'a large dataset', 'clinical dataset'); if the paper only describes the data generically, leave this field empty.\n\n"
            f"Text:\n{context}\n\n"
            "Output Instructions:\n"
            "- Return valid JSON only, with a single field 'datasets' that is a list of dataset names, e.g., {\"datasets\": [\"FINA09\",\"MTL-AQA\"]}.\n"
            "- If multiple datasets are mentioned, include all of them in the same list.\n"
            "- If no datasets meet these conditions, return an empty list.\n"
        )

        _rate_limit_block_until_allowed()

        backoff = 5
        extracted = []

        # Retry logic
        for attempt in range(4):
            try:
                _sent_calls_count += 1
                resp = client.models.generate_content(
                    model=MODEL,
                    contents=prompt,
                    config={
                        "response_mime_type": "application/json",
                        "response_schema": ExtractedDatasets,
                        "temperature": 0.0,
                        "top_k": 1,
                        "top_p": 1.0
                    },
                )

                # Token tracking
                if resp.usage_metadata:
                    _total_input_tokens += resp.usage_metadata.prompt_token_count
                    _total_output_tokens += resp.usage_metadata.candidates_token_count

                extracted = resp.parsed.datasets or []
                extracted = [s.strip() for s in extracted if s and s.strip()]
                break

            except Exception as e:
                print(f"[LLM ERROR] paper={source} attempt={attempt+1} error={e}")
                time.sleep(backoff)
                backoff *= 2

        paper_datasets[source] = extracted
        print(f"[EXTRACT] {source}: found {len(extracted)} datasets")

    # Prepare output table
    all_pdf_files = [f for f in os.listdir(DOCUMENT_FOLDER) if f.endswith(".pdf")]

    final_list = []
    for pdf in all_pdf_files:
        datasets = paper_datasets.get(pdf, [])
        datasets_str = ";".join(sorted(set(datasets))) if datasets else ""
        final_list.append({"Title": pdf, "Data Source": datasets_str})

    return final_list


# Execution
start_index_time = time.monotonic()
vectorstore, retriever, total_chunks = build_or_load_index(DOCUMENT_FOLDER, VECTOR_DB_PATH)
end_index_time = time.monotonic()
total_index_time_seconds = end_index_time - start_index_time

query = "Human Action Quality Assessment (AQA) datasets"

start_total_time = time.monotonic()
extracted = extract_datasets_per_pdf(retriever, query, total_chunks=total_chunks, top_chunks_per_paper=5)
end_total_time = time.monotonic()
total_extract_time_seconds = end_total_time - start_total_time

total_pipeline_time_seconds = total_index_time_seconds + total_extract_time_seconds

df_results = pd.DataFrame(extracted)
print("[DONE] Extraction completed.")

[LOAD] An et al. - 2024 - Multi-Stage Contrastive Regression for Action Quality Assessment.pdf: 6 pages
[LOAD] Bai et al. - 2022 - Action Quality Assessment with Temporal Parsing Transformer.pdf: 17 pages
[LOAD] Cao et al. - 2024 - Action-Evaluator A Visualization Approach for Player Action Evaluation in Soccer.pdf: 11 pages
[LOAD] Chariar et al. - 2023 - AI Trainer Autoencoder Based Approach for Squat Analysis and Correction.pdf: 16 pages
[LOAD] Chen - 2022 - A Data Mining‐Based Model for Evaluating Tennis Players’ Training Movements.pdf: 11 pages
[LOAD] Chen et al. - 2021 - SportsCap Monocular 3D Human Motion Capture and Fine-Grained Understanding in Challenging Sports Vi.pdf: 19 pages
[LOAD] Hipiny et al. - 2023 - Who danced better Ranked tiktok dance video dataset and pairwise action quality assessment method.pdf: 12 pages
[LOAD] Jain et al. - 2021 - Action Quality Assessment Using Siamese Network-Based Deep Metric Learning.pdf: 14 pages
[LOAD] Jin et al. - 2015 - Virtual Personal 

In [13]:
# --- FINAL REPORT AND COSTS ---

theoretical_min_time = ((_sent_calls_count - 1) // _RATE_MAX_CALLS) * _RATE_WINDOW_SEC if _sent_calls_count > 0 else 0

print("\n" + "="*40)
print("EXECUTION REPORT - DATASET EXTRACTION")
print("="*40)

print(f"Papers Processed: {len(extracted)}")
print(f"API Calls made: {_sent_calls_count}")
print(f"Input Tokens consumed: {_total_input_tokens}")
print(f"Output Tokens generated: {_total_output_tokens}")
print(f"Embedding Tokens: {_total_embedding_tokens}")
print(f"Execution Time: {total_pipeline_time_seconds/60:.2f} min (Theoretical: {theoretical_min_time/60:.2f} min)")

# Pricing
PRICE_PER_1M_INPUT_TOKENS = 0.3
PRICE_PER_1M_OUTPUT_TOKENS = 2.5
PRICE_PER_1M_EMBEDDING_TOKENS = 0.20

cost_input = (_total_input_tokens / 1_000_000) * PRICE_PER_1M_INPUT_TOKENS
cost_output = (_total_output_tokens / 1_000_000) * PRICE_PER_1M_OUTPUT_TOKENS
cost_embedding = (_total_embedding_tokens / 1_000_000) * PRICE_PER_1M_EMBEDDING_TOKENS
cost_total = cost_input + cost_output + cost_embedding

print("-" * 40)
print(f"Input Cost:  ${cost_input:.5f}")
print(f"Output Cost: ${cost_output:.5f}")
print(f"Embedding Cost: ${cost_embedding:.5f}")
print(f"Estimated Total Cost: ${cost_total:.5f}")
print("="*40)


EXECUTION REPORT - DATASET EXTRACTION
Papers Processed: 21
API Calls made: 20
Input Tokens consumed: 46083
Output Tokens generated: 219
Embedding Tokens: 302586
Execution Time: 2.72 min (Theoretical: 1.00 min)
----------------------------------------
Input Cost:  $0.01382
Output Cost: $0.00055
Embedding Cost: $0.06052
Estimated Total Cost: $0.07489


In [14]:
folder_path = '../data/'
result_path = folder_path + 'results/'
ELIGIBILITY_DIR = result_path + 'eligibility'

def append_run_summary(filepath, row_dict):
    file_exists = os.path.isfile(filepath)
    with open(filepath, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=row_dict.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_dict)

append_run_summary(
    f"{ELIGIBILITY_DIR}/runs_cost_summary.csv",
    {
        "run_id": RUN_ID,
        "stage": "eligibility",
        "model": MODEL,
        "records_processed": len(df_results),
        "api_calls": _sent_calls_count,
        "input_tokens": _total_input_tokens,
        "output_tokens": _total_output_tokens,
        "embedding_tokens": _total_embedding_tokens,
        "index_time_sec": total_index_time_seconds,
        "extraction_time_sec": total_extract_time_seconds,
        "execution_time_sec": total_pipeline_time_seconds,
        "cost_input_usd": cost_input,
        "cost_output_usd": cost_output,
        "cost_embedding_usd": cost_embedding,
        "cost_usd": cost_total,
    }
)

In [15]:
# Harmonization

_harm_calls_count = 0
_harm_input_tokens = 0
_harm_output_tokens = 0

def harmonize_all_data_sources(df: pd.DataFrame, client, model=None, max_retries: int = 3):
    """Harmonizes dataset names across all rows using an LLM."""
    global _harm_calls_count, _harm_input_tokens, _harm_output_tokens

    if model is None:
        model = MODEL

    # Prepare records for the harmonization prompt
    records = []
    for idx, row in df.iterrows():
        sources = "" if pd.isna(row.get("Data Source")) else str(row.get("Data Source"))
        records.append(f"- id: {int(idx)}\n  Data Source: {sources}")

    prompt = (
        "Check and harmonize all dataset names for consistency and correctness.\n"
        "Guidelines:\n"
        "- Remove descriptors like 'dataset', 'database', 'data'.\n"
        "- If multiple datasets are listed, leave the separation with ';' (semicolon) without spaces between the datasets.\n"
        "- Always capitalize the first letter of each dataset.\n"
        "- Prefer the recognized abbreviation if it exists, otherwise use the full name (e.g., 'Massachusetts Institute of Technology' becomes 'MIT','Piano Skills Assessment' becomes 'PISA').\n"
        "- If a field is empty, keep it empty.\n\n"
        "Output Instructions:\n"
        "- Return only JSON matching the schema.\n"
        "- Return exactly one harmonized string per input row, in the same order as provided.\n\n"
        "Records to harmonize:\n"
        + "\n".join(records)
    )

    # Retry mechanism for robustness
    backoff = 5
    for attempt in range(max_retries):
        _rate_limit_block_until_allowed()
        try:
            _harm_calls_count += 1
            resp = client.models.generate_content(
                model=model,
                contents=prompt,
                config={
                    "response_mime_type": "application/json",
                    "response_schema": HarmonizedSources,
                    "temperature": 0.0,
                    "top_k": 1,
                    "top_p": 1.0
                },
            )
            if resp.usage_metadata:
                _harm_input_tokens += resp.usage_metadata.prompt_token_count
                _harm_output_tokens += resp.usage_metadata.candidates_token_count

            harmonized = resp.parsed.sources

            # Ensure output length matches input length
            if len(harmonized) != len(df):
                harmonized = (harmonized + [""] * len(df))[:len(df)]
            return harmonized

        except Exception as e:
            print(f"[HARMONIZE ERROR] attempt={attempt+1} error={e}")
            time.sleep(backoff)
            backoff *= 2

    # Fallback: return original values unchanged
    fallback = []
    for _, row in df.iterrows():
        s = "" if pd.isna(row.get("Data Source")) else str(row.get("Data Source"))
        fallback.append(s)

    return fallback


# Apply harmonization to dataframe
start_harm_time = time.monotonic()
df_results["Data Source Harmonized"] = harmonize_all_data_sources(df_results, client, model=MODEL_SUPERVISOR)
end_harm_time = time.monotonic()
total_harm_time_seconds = end_harm_time - start_harm_time

df_results

,Title,Data Source,Data Source Harmonized
0,An et al. - 2024 - Multi-Stage Contrastive Reg...,FineDiving,FineDiving
1,Bai et al. - 2022 - Action Quality Assessment ...,AQA-7;JIGSAWS;MTL-AQA,AQA-7;JIGSAWS;MTL-AQA
2,Cao et al. - 2024 - Action-Evaluator A Visuali...,,
3,Chariar et al. - 2023 - AI Trainer Autoencoder...,,
4,Chen - 2022 - A Data Mining‐Based Model for Ev...,,
5,Chen et al. - 2021 - SportsCap Monocular 3D Hu...,MTL-AQA;SMART,MTL-AQA;SMART
6,Hipiny et al. - 2023 - Who danced better Ranke...,,
7,Jain et al. - 2021 - Action Quality Assessment...,Gymnastic vaults;MTL-AQA;Olympics Diving,Gymnastic Vaults;MTL-AQA;Olympics Diving
8,Jin et al. - 2015 - Virtual Personal Trainer v...,,
9,Joung et al. - 2023 - Contrastive Learning for...,AI-Hub Fitness Dataset;Squat Dataset,AI-Hub Fitness;Squat


In [16]:
# --- FINAL REPORT AND COSTS (HARMONIZATION) ---

print("\n" + "="*40)
print("EXECUTION REPORT - HARMONIZATION")
print("="*40)

print(f"API Calls made: {_harm_calls_count}")
print(f"Input Tokens consumed: {_harm_input_tokens}")
print(f"Output Tokens generated: {_harm_output_tokens}")

# Pricing GEMINI 2.5 Pro
PRICE_PER_1M_INPUT_TOKENS = 1.25
PRICE_PER_1M_OUTPUT_TOKENS = 10

cost_input = (_harm_input_tokens / 1_000_000) * PRICE_PER_1M_INPUT_TOKENS
cost_output = (_harm_output_tokens / 1_000_000) * PRICE_PER_1M_OUTPUT_TOKENS
cost_total = cost_input + cost_output

print("-" * 40)
print(f"Input Cost:  ${cost_input:.5f}")
print(f"Output Cost: ${cost_output:.5f}")
print(f"Estimated Total Cost: ${cost_total:.5f}")
print("="*40)


EXECUTION REPORT - HARMONIZATION
API Calls made: 1
Input Tokens consumed: 506
Output Tokens generated: 187
----------------------------------------
Input Cost:  $0.00063
Output Cost: $0.00187
Estimated Total Cost: $0.00250


In [17]:
append_run_summary(
    f"{ELIGIBILITY_DIR}/runs_cost_summary.csv",
    {
        "run_id": RUN_ID,
        "stage": "harmonization",
        "model": MODEL_SUPERVISOR,
        "records_processed": len(df_results),
        "api_calls": _harm_calls_count,
        "input_tokens": _harm_input_tokens,
        "output_tokens": _harm_output_tokens,
        "embedding_tokens": None,
        "index_time_sec": None,
        "extraction_time_sec": None,
        "execution_time_sec": total_harm_time_seconds,
        "cost_input_usd": cost_input,
        "cost_output_usd": cost_output,
        "cost_embedding_usd": None,
        "cost_usd": cost_total,
    }
)

In [18]:
df_results.to_csv(f"{ELIGIBILITY_DIR}/{RUN_ID}_records.csv", index=False)

In [12]:
# Final DataFrame Preparation
df_evaluated_final = df_results.copy()
df_evaluated_final["Data Source"] = df_evaluated_final["Data Source Harmonized"]
df_evaluated_final.drop(columns=["Data Source Harmonized"], inplace=True)

# Clean and normalize the Title field
df_evaluated_final["Title"] = (
    df_evaluated_final["Title"]
    .astype(str)
    .str.replace(r"^.* - ", "", regex=True)
    .str.replace(r"\.pdf$", "", flags=re.IGNORECASE, regex=True)
    .str.strip()
)
df_evaluated_final

,Title,Data Source
0,Multi-Stage Contrastive Regression for Action ...,FineDiving
1,Action Quality Assessment with Temporal Parsin...,AQA-7;JIGSAWS;MTL-AQA
2,Action-Evaluator A Visualization Approach for ...,
3,AI Trainer Autoencoder Based Approach for Squa...,Countix;FitnessAQA;Kinetics 700;MM-Fit;Single ...
4,A Data Mining‐Based Model for Evaluating Tenni...,
5,SportsCap Monocular 3D Human Motion Capture an...,AQA;Diving48;FineGym;MTL-AQA;SMART
6,Who danced better Ranked tiktok dance video da...,Ranked TikTok
7,Action Quality Assessment Using Siamese Networ...,MIT-Diving;MTL-AQA;UNLV-Diving
8,Virtual Personal Trainer via the Kinect Sensor,
9,Contrastive Learning for Action Assessment Usi...,AI-Hub Fitness;Squat


In [13]:
# Merge abstract metadata with the harmonized dataset sources
folder_path = '../data/'
result_path = folder_path + 'results/'

df_abstract = pd.read_csv(result_path + 'df_evaluated_validation_abstract.csv')

# Sort both tables by Title for proper alignment
df_abstract_sorted = df_abstract.sort_values(by="Title").reset_index(drop=True)
df_evaluated_final_sorted = df_evaluated_final.sort_values(by="Title").reset_index(drop=True)

# Combine metadata with dataset information
df_review = pd.DataFrame({
    "DOI": df_abstract_sorted["DOI"],
    "Title": df_abstract_sorted["Title"],
    "Authors": df_abstract_sorted["Authors"],
    "Year": df_abstract_sorted["Year"],
    "Include": df_abstract_sorted["Include"],
    "Data Source": df_evaluated_final_sorted["Data Source"]
})

# Remove papers without datasets
df_review = df_review[df_review["Data Source"] != ""].reset_index(drop=True)
df_review = df_review.drop(columns=["Include"], errors="ignore")
df_review

,DOI,Title,Authors,Year,Data Source
0,10.1109/ACCESS.2023.3316009,AI Trainer: Autoencoder Based Approach for Squ...,M. Chariar; S. Rao; A. Irani; S. Suresh; C. S....,2023,Countix;FitnessAQA;Kinetics 700;MM-Fit;Single ...
1,10.1109/TCSVT.2020.3017727,Action Quality Assessment Using Siamese Networ...,"Jain, Hiteshi; Harit, Gaurav; Sharma, Avinash",2021,MIT-Diving;MTL-AQA;UNLV-Diving
2,10.1109/ICMLC58545.2023.10327994,Action Quality Assessment for ASD Behaviour Ev...,D. Zhang; D. Zhou; H. Liu,2023,AQA-7;BEST;DREAM;EPIC-Skills;Fis-V;Infinite Gr...
3,10.1007/978-3-031-19772-7_25,Action Quality Assessment with Temporal Parsin...,"Bai, Y.; Zhou, D.; Zhang, S.; Wang, J.; DIng, ...",2022,AQA-7;JIGSAWS;MTL-AQA
4,10.1109/TMM.2023.3294800,Adaptive Stage-Aware Assessment Skill Transfer...,S. -J. Zhang; J. -H. Pan; J. Gao; W. -S. Zheng,2024,AQA-7;BEST;EPIC-Skills;JIGSAWS
5,10.1109/ACCESS.2023.3305372,Contrastive Learning for Action Assessment Usi...,C. -I. Joung; S. Byun; S. Baek,2023,AI-Hub Fitness;Squat
6,10.1109/WACV48630.2021.00044,EAGLE-Eye: Extreme-pose Action Grader using de...,M. Nekoui; F. O. Tito Cruz; L. Cheng,2021,AQA-7;ExPose;G-ExPose;MIT-Skate
7,10.1109/ICMEW46912.2020.9106049,Efficient Fitness Action Analysis Based on Spa...,J. Li; H. Cui; T. Guo; Q. Hu; Y. Shen,2020,Fitness Action
8,10.1007/978-3-030-00767-6_12,End-to-end learning for action quality assessment,"Li, Y.; Chai, X.; Chen, X.",2018,MIT-Skating;UNLV-Diving;UNLV-Vault
9,10.1109/CVPRW.2017.16,Learning to Score Olympic Events,P. Parmar; B. T. Morris,2017,FINA09


In [14]:
# Map partial dataset names to standardized identifiers
dataset_map = {
    "AQA": ["AQA-7"],
    "MIT-Skating": ["MIT-Skate"],
    "MIT-Diving": ["MIT-Dive"],
    "MIT-Skiing": ["MIT-Skate"],
    "Olympics Diving": ["MIT-Dive"],
    "MIT Olympic Scoring": ["MIT-Skate", "MIT-Dive"],
    "Olympic sports": ["MIT-Skate", "MIT-Dive"],
    "UNLV-Skating": ["UNLV-Skate"],
    "UNLV-Diving": ["UNLV-Dive"],
    "Vault": ["UNLV-Vault"],
    "Gymnastic vaults": ["UNLV-Vault"],
    "UNLV Olympic Scoring": ["UNLV-Skate", "UNLV-Dive", "UNLV-Vault"],
    "FineDiving-HM": ["FineDiving"],
    "Fis-V": ["FIS-V"],
    "FisV-5": ["FIS-V"],
    "IntelliRehabDS": ["IRDS"],
    "Multitask-AQA": ["MTL-AQA"],
    "Fitness Action": ["Fitness-AQA"],
    "FitnessAQA": ["Fitness-AQA"],
    "RG": ["Rhythmic Gymnastics"],
    "MLT-AQA": ["MTL-AQA"],
    "AFG-Olympics": ["AGF-Olympics"],
    "Epic Skills 2018": ["EPIC-Skills"],
    "MIT-Diving & Skiing": ["MIT-Dive"],
    "Kimore": ["KIMORE"],
    "G-ExPose": ["ExPose"]
}

def map_partial_sources(cell):
    """Replace partial or alternative dataset names with standardized identifiers."""
    if pd.isna(cell):
        return cell

    # Split multiple datasets by semicolon
    parts = [p.strip() for p in str(cell).split(";") if p.strip()]
    out = []

    for p in parts:
        matched = False
        for k, v in dataset_map.items():
            pattern = fr"(^|;){re.escape(k)}($|;)"
            if re.fullmatch(k, p):
                out.extend(v)
                matched = True
                break
            elif re.search(pattern, ";" + p + ";"):
                out.extend(v)
                matched = True
                break
        if not matched:
            out.append(p)

    # Remove duplicates and limit length
    seen = set()
    result = []
    for x in out:
        key = x.strip()
        if key not in seen:
            seen.add(key)
            result.append(key)

    result = [x for x in result if len(x) <= 20]
    return ";".join(result)

# Apply mapping to harmonize dataset names
df_review["Data Source"] = df_review["Data Source"].apply(map_partial_sources)
df_review

,DOI,Title,Authors,Year,Data Source
0,10.1109/ACCESS.2023.3316009,AI Trainer: Autoencoder Based Approach for Squ...,M. Chariar; S. Rao; A. Irani; S. Suresh; C. S....,2023,Countix;Fitness-AQA;Kinetics 700;MM-Fit;UCF-101
1,10.1109/TCSVT.2020.3017727,Action Quality Assessment Using Siamese Networ...,"Jain, Hiteshi; Harit, Gaurav; Sharma, Avinash",2021,MIT-Dive;MTL-AQA;UNLV-Dive
2,10.1109/ICMLC58545.2023.10327994,Action Quality Assessment for ASD Behaviour Ev...,D. Zhang; D. Zhou; H. Liu,2023,AQA-7;BEST;DREAM;EPIC-Skills;FIS-V;Infinite Gr...
3,10.1007/978-3-031-19772-7_25,Action Quality Assessment with Temporal Parsin...,"Bai, Y.; Zhou, D.; Zhang, S.; Wang, J.; DIng, ...",2022,AQA-7;JIGSAWS;MTL-AQA
4,10.1109/TMM.2023.3294800,Adaptive Stage-Aware Assessment Skill Transfer...,S. -J. Zhang; J. -H. Pan; J. Gao; W. -S. Zheng,2024,AQA-7;BEST;EPIC-Skills;JIGSAWS
5,10.1109/ACCESS.2023.3305372,Contrastive Learning for Action Assessment Usi...,C. -I. Joung; S. Byun; S. Baek,2023,AI-Hub Fitness;Squat
6,10.1109/WACV48630.2021.00044,EAGLE-Eye: Extreme-pose Action Grader using de...,M. Nekoui; F. O. Tito Cruz; L. Cheng,2021,AQA-7;ExPose;MIT-Skate
7,10.1109/ICMEW46912.2020.9106049,Efficient Fitness Action Analysis Based on Spa...,J. Li; H. Cui; T. Guo; Q. Hu; Y. Shen,2020,Fitness-AQA
8,10.1007/978-3-030-00767-6_12,End-to-end learning for action quality assessment,"Li, Y.; Chai, X.; Chen, X.",2018,MIT-Skate;UNLV-Dive;UNLV-Vault
9,10.1109/CVPRW.2017.16,Learning to Score Olympic Events,P. Parmar; B. T. Morris,2017,FINA09


In [15]:
# Load already reviewed data based on abstracts
df_review_abstract = pd.read_csv(result_path + 'review.csv')
df_review_abstract

,DOI,Title,Authors,Year,Data Source
0,10.1109/ACCESS.2024.3423462,MMW-AQA: Multimodal In-the-Wild Dataset for Ac...,T. Nagai; S. Takeda; S. Suzuki; H. Seshimo,2024,MMW-AQA
1,10.1109/ICAACE61206.2024.10548995,Action Quality Assessment with Multi-scale Tem...,W. Wang; H. Wang; Y. Hao; Q. Wang,2024,AQA-7;MTL-AQA
2,10.1109/ITME53901.2021.00048,Skeleton Based Action Quality Assessment of Fi...,H. -Y. Li; Q. Lei; H. -B. Zhang; J. -X. Du,2021,MIT-Skate
3,10.1109/CVPR52729.2023.00238,LOGO: A Long-Form Video Dataset for Group Acti...,S. Zhang; W. Dai; S. Wang; X. Shen; J. Lu; J. ...,2023,LOGO
4,10.1109/TIM.2024.3398072,Learning Sparse Temporal Video Mapping for Act...,S. Zahan; G. Mubashar Hassan; A. Mian,2024,AGF-Olympics
...,...,...,...,...,...
70,10.1109/SSCI.2017.8285270,The open online repository of karate motion ca...,"Hachaj, T.; Ogiela, M.R.; Piekarczyk, M.",2017,self-created
71,10.1007/978-3-319-10599-4_36,Assessing the quality of actions,"Pirsiavash, H.; Vondrick, C.; Torralba, A.",2014,MIT-Skate;MIT-Dive
72,10.1007/s00521-023-09068-w,Auto-encoding score distribution regression fo...,"Zhang, Boyu; Chen, Jiayuan; Xu, Yinfei; Zhang,...",2023,AQA-7;MTL-AQA;JIGSAWS
73,10.1038/s41597-022-01188-7,Functional movement screen dataset collected w...,"Xing, Qing-Jun; Shen, Yuan-Yuan; Cao, Run; Zon...",2022,self-created


In [16]:
# Combine all review data and export final CSV
df_review_final = pd.concat([df_review, df_review_abstract], ignore_index=True)
df_review_final.to_csv(result_path + 'review_final.csv', index=False)
df_review_final

,DOI,Title,Authors,Year,Data Source
0,10.1109/ACCESS.2023.3316009,AI Trainer: Autoencoder Based Approach for Squ...,M. Chariar; S. Rao; A. Irani; S. Suresh; C. S....,2023,Countix;Fitness-AQA;Kinetics 700;MM-Fit;UCF-101
1,10.1109/TCSVT.2020.3017727,Action Quality Assessment Using Siamese Networ...,"Jain, Hiteshi; Harit, Gaurav; Sharma, Avinash",2021,MIT-Dive;MTL-AQA;UNLV-Dive
2,10.1109/ICMLC58545.2023.10327994,Action Quality Assessment for ASD Behaviour Ev...,D. Zhang; D. Zhou; H. Liu,2023,AQA-7;BEST;DREAM;EPIC-Skills;FIS-V;Infinite Gr...
3,10.1007/978-3-031-19772-7_25,Action Quality Assessment with Temporal Parsin...,"Bai, Y.; Zhou, D.; Zhang, S.; Wang, J.; DIng, ...",2022,AQA-7;JIGSAWS;MTL-AQA
4,10.1109/TMM.2023.3294800,Adaptive Stage-Aware Assessment Skill Transfer...,S. -J. Zhang; J. -H. Pan; J. Gao; W. -S. Zheng,2024,AQA-7;BEST;EPIC-Skills;JIGSAWS
...,...,...,...,...,...
86,10.1109/SSCI.2017.8285270,The open online repository of karate motion ca...,"Hachaj, T.; Ogiela, M.R.; Piekarczyk, M.",2017,self-created
87,10.1007/978-3-319-10599-4_36,Assessing the quality of actions,"Pirsiavash, H.; Vondrick, C.; Torralba, A.",2014,MIT-Skate;MIT-Dive
88,10.1007/s00521-023-09068-w,Auto-encoding score distribution regression fo...,"Zhang, Boyu; Chen, Jiayuan; Xu, Yinfei; Zhang,...",2023,AQA-7;MTL-AQA;JIGSAWS
89,10.1038/s41597-022-01188-7,Functional movement screen dataset collected w...,"Xing, Qing-Jun; Shen, Yuan-Yuan; Cao, Run; Zon...",2022,self-created


In [17]:
# Compare DOIs between study dataset and validation set
with open(folder_path + 'study_datasets.json', 'r') as f:
    study_papers_list = json.load(f)

verified_dois_study = set(study_papers_list.keys())
verified_dois_validation = set(df_review_final["DOI"].tolist())

matches = verified_dois_study & verified_dois_validation
num_matches = len(matches)

only_study = verified_dois_study - verified_dois_validation
num_study = len(only_study)

only_validation = verified_dois_validation - verified_dois_study
num_validation = len(only_validation)

print("Perfect Matches:", num_matches)
print("Only in Study:", num_study)
print("Only in Validation:", num_validation)

Perfect Matches: 67
Only in Study: 27
Only in Validation: 24


In [18]:
# Compare dataset names between validation and study
filtered_sources = df_review_final['Data Source'].dropna().unique().tolist()
filtered_sources = [source.strip() for item in filtered_sources for source in item.split(';')]
filtered_sources = set(filtered_sources)
filtered_sources.discard("self-created")

# Extract verified datasets from the study JSON
verified_datasets_study = set()
for value in study_papers_list.values():
    if value:
        parts = [v.strip() for v in value.split(';') if v.strip()]
        verified_datasets_study.update(parts)
verified_datasets_study.discard("self-created")

# Compute intersections and differences
common_elements = filtered_sources & verified_datasets_study
only_in_filtered = filtered_sources - verified_datasets_study
only_in_original = verified_datasets_study - filtered_sources


print("Elements in common:", common_elements)
print("Number of elements in common:", len(common_elements))

print("\nElements in Validation but not in Study:", only_in_filtered)
print("Number:", len(only_in_filtered))

print("\nElements in Study but not in Validation:", only_in_original)
print("Number:", len(only_in_original))

Elements in common: {'NETS', 'FIS-V', 'UI-PRMD', 'MIT-Dive', 'FineFS', 'TaiChi-24', 'Walking Gait', 'Origami Video', 'Squat', 'EHE', 'JDM', 'KIMORE', 'UNLV-Vault', 'OlympicFS', 'TASD-2', 'MMASD', 'BEST', 'MMW-AQA', 'PaSk', 'PT13', 'Fri2023', 'UNLV-Skate', 'AQA-7', 'PISA', 'RFSJ', 'FineDiving', 'Fitness-28', 'JIGSAWS', 'FineGym', 'DREAM', 'IRDS', 'PD4T', 'UNLV-Dive', 'Rhythmic Gymnastics', 'LOGO', 'UMONS-TAICHI', 'MTL-AQA', 'EPIC-Skills', 'AI-Hub Fitness', 'MIT-Skate'}
Number of elements in common: 40

Elements in Validation but not in Study: {'SMART', 'Thumos14', 'Diving48', 'ActivityNet1.3', 'FINA09', 'KTH', 'Golf Swing Video', 'CAD-60', 'RDSD', 'Northwestern-UCLA', 'ExPose', 'AGF-Olympics', 'Kinetics 700', 'Countix', 'FineRehab', 'Golf Swing Action', 'Infinite Grasp', '3D Animation', 'UCF-101', 'Ranked TikTok', 'MM-Fit', 'MSR-Action3D', 'TAQR', 'Fitness-AQA', 'MSRC-12'}
Number: 25

Elements in Study but not in Validation: {'SkatingVerse', 'SU-41 Gesture', 'CPR-Coach', 'Stroke Rehabil